# 🎓 Day 1 · Session 5
# 05C · Advanced RAG & Debugging
## Chunk tuning, top-k, prompt hardening, evaluation and security

## 🎯 Learning Objectives

- Tune retrieval quality
- Debug RAG failures
- Apply production RAG checklist

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas numpy scikit-learn pypdf chromadb langchain langchain-openai langchain-community langchain-chroma

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
import numpy as np
from openai import OpenAI

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.2, system_prompt="You are a helpful academic assistant.", max_tokens=700):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ Advanced RAG

Basic RAG works, but production RAG needs tuning.

Common problems:
- wrong chunks retrieved
- chunks too small
- chunks too large
- answer not grounded
- missing citations
- poor out-of-scope handling

In [ ]:
rag_debug_checklist = pd.DataFrame([
    {"Problem": "Wrong answer", "Check First": "Were the right chunks retrieved?"},
    {"Problem": "Answer too vague", "Check First": "Is chunk size too small?"},
    {"Problem": "Answer has irrelevant detail", "Check First": "Is chunk size too large or top_k too high?"},
    {"Problem": "Hallucination", "Check First": "Does prompt say ONLY use context?"},
    {"Problem": "No citation", "Check First": "Are sources passed in context?"},
    {"Problem": "Missing answer", "Check First": "Was document indexed correctly?"}
])
rag_debug_checklist

# 2️⃣ Chunk Size Tuning

In [ ]:
sample_text = '''
M.Tech CSE Fee Structure:
The tuition fee for M.Tech CSE is Rs. 50,000 per semester.
The programme duration is 4 semesters.
GATE-qualified students are eligible for a 50% tuition scholarship.

NLP Elective:
The NLP elective is offered in Semester 3.
Prerequisites: Machine Learning and Python Programming.
Faculty Coordinator: Dr. Kavitha Raman.

Attendance Rule:
Students must maintain 75% attendance to be eligible for final examinations.
'''

def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

for size in [100, 250, 500]:
    ch = chunk_text(sample_text, size, 50)
    print(f"Chunk size {size}: {len(ch)} chunks")
    for i, c in enumerate(ch, 1):
        print(f"Chunk {i}: {c[:100].replace(chr(10), ' ')}...")
    print("-" * 100)

## Rule of Thumb

- Small chunks: precise but may lose context
- Large chunks: more context but may retrieve noise
- Start with 400–800 characters for small demos
- Use 100–150 character overlap

# 3️⃣ Top-K Tuning

In [ ]:
topk_table = pd.DataFrame([
    {"top_k": 1, "Pros": "Focused, low cost", "Cons": "May miss needed context"},
    {"top_k": 3, "Pros": "Good default", "Cons": "Slightly more tokens"},
    {"top_k": 5, "Pros": "More context", "Cons": "Can add noise"},
    {"top_k": 10, "Pros": "Useful for broad questions", "Cons": "High cost and distraction"}
])
topk_table

# 4️⃣ Similarity Threshold

In [ ]:
threshold_table = pd.DataFrame([
    {"Similarity Score": "> 0.85", "Interpretation": "Highly relevant"},
    {"Similarity Score": "0.75–0.85", "Interpretation": "Possibly relevant"},
    {"Similarity Score": "0.60–0.75", "Interpretation": "Weak relevance"},
    {"Similarity Score": "< 0.60", "Interpretation": "Likely irrelevant"}
])
threshold_table

# 5️⃣ RAG Prompt Hardening

In [ ]:
strict_rag_prompt = '''
You are a careful academic assistant.

Rules:
1. Answer ONLY from the provided context.
2. If the answer is not present, say: "I don't have this information in the provided knowledge base."
3. Always mention the source.
4. Do not use outside knowledge.
5. Do not guess.
6. Keep the answer concise.

Context:
{context}

Question:
{question}
'''
print(strict_rag_prompt)

# 6️⃣ RAG Evaluation

In [ ]:
eval_questions = pd.DataFrame([
    {"Question": "What is the M.Tech CSE fee?", "Expected Source": "department_handbook.txt", "Expected Behavior": "Answer with fee"},
    {"Question": "Who coordinates NLP?", "Expected Source": "course_catalog.txt", "Expected Behavior": "Answer with faculty name"},
    {"Question": "What is the hostel fee?", "Expected Source": "None", "Expected Behavior": "Say not available"},
    {"Question": "When are lab records submitted?", "Expected Source": "lab_manual.txt", "Expected Behavior": "Answer Friday"}
])
eval_questions

## Evaluation Metrics

For beginner RAG systems, track:

1. Did retrieval return the correct chunk?
2. Did the answer use only retrieved context?
3. Did it cite the source?
4. Did it refuse out-of-scope questions?
5. Was the answer concise and useful?

# 7️⃣ Metadata Filtering

In [ ]:
metadata_example = pd.DataFrame([
    {"Chunk": "M.Tech fee details", "source": "department_handbook.txt", "doc_type": "handbook", "semester": "all"},
    {"Chunk": "NLP prerequisites", "source": "course_catalog.txt", "doc_type": "catalog", "semester": "3"},
    {"Chunk": "Lab submission rules", "source": "lab_manual.txt", "doc_type": "lab", "semester": "all"}
])
metadata_example

Metadata lets you filter retrieval.

Examples:
- retrieve only Semester 3 documents
- retrieve only lab manuals
- retrieve only approved policy documents
- retrieve only current-year documents

# 8️⃣ RAG Security

In [ ]:
security_checklist = pd.DataFrame([
    {"Risk": "Sensitive documents indexed", "Control": "Review documents before indexing"},
    {"Risk": "Vector DB exposed publicly", "Control": "Use authentication and network restrictions"},
    {"Risk": "Prompt injection in documents", "Control": "Sanitize and validate retrieved chunks"},
    {"Risk": "Wrong access control", "Control": "Filter documents by user permissions"},
    {"Risk": "Data leakage in logs", "Control": "Mask sensitive prompts and responses"}
])
security_checklist

# 9️⃣ RAG Debugger Flow

In [ ]:
rag_debugger = '''
RAG answer is wrong
        |
        v
Were the correct chunks retrieved?
        |
   YES  |  NO
   v        v
Check     Improve chunking,
prompt    embeddings, metadata,
grounding top_k, document quality
        |
        v
Does answer cite source?
        |
   YES  |  NO
   v        v
Evaluate Add citation instruction
quality   and source metadata
'''
print(rag_debugger)

# 🔟 AI Pulse: Protect Your Vector Store

Your vector database contains your institutional knowledge.

Treat it like source code or confidential documents.

Ask:

> What is more valuable in an AI system: the model or the knowledge base?

# 🧪 Lab

Tune a RAG system by changing chunk size, overlap, top_k, prompt strictness and citation format.

# ✅ Summary

Production RAG needs retrieval tuning, strict prompts, citations, evaluation and security controls.